# Plotting the skeletons of every neuron in a module 

In [1]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import bokeh
from bokeh.plotting import figure, output_notebook, show, output_file, gridplot
from bokeh.io import export_svg, export_png
from neuprint import Client

auth_token_file = open("flybrain.auth.txt", 'r')
auth_token = next(auth_token_file).strip()
try:
    np_client = Client('neuprint.janelia.org', dataset='hemibrain:v1.2.1', token=auth_token)
except:
    np_client = None

output_notebook()

Loading BokehJS ...

In [2]:
ovi_node_df  = pd.read_csv('modularity_runs/0.0/0-0_98765.txt', header=None, sep=' ', names=['id', "0.0"])
ovi_clusters = ovi_node_df[['id', '0.0']].copy()
ovi_clusters

,id,0.0
0,1003215282,1
1,1005952640,2
2,1006928515,3
3,1007260806,3
4,1008024276,4
...,...,...
1827,987117151,2
1828,987273073,6
1829,988567837,5
1830,988909130,5


In [5]:
ovi_clusters['0.0'].value_counts()

0.0
4    347
3    319
1    309
7    231
5    217
6    210
2    199
Name: count, dtype: int64

In [3]:
# map on colors
color_dict = {1: '#4e90d3', 2: '#9467bd', 3: '#e7cf57', 4: '#ff6a88', 5: '#5cc9ff', 6: '#3a9f82', 7: '#9fad2b', 8: '#ff7f0e'}
ovi_clusters['color'] = ovi_clusters['0.0'].map(color_dict)

In [6]:
# create a function to choose which module to return all skeletons for
def get_and_plot(synapse_plot, cluster_id):
    cluster_curr = synapse_plot[synapse_plot['0.0']== cluster_id]
    
    # get all skeletons 
    all_skeletons = []
    # cretae skeleton for both oviINs
    for id in cluster_curr['id'].unique():
        skeletons = []
        s = np_client.fetch_skeleton(id, format='pandas')
        s['bodyId'] = id
        s['color'] = synapse_plot[synapse_plot['id']==id]['color'].values[0] # Use module color
        skeletons.append(s)

        skeletons = pd.concat(skeletons, ignore_index=True)
        # Join parent nodes
        segments = skeletons.merge(skeletons, left_on=['bodyId', 'link'], right_on=['bodyId', 'rowId'], suffixes=['_child', '_parent'])
        all_skeletons.append(segments)

    return all_skeletons

In [7]:
# get skeletons for by cluster
skel = get_and_plot(ovi_clusters, 2)
skel[:2]

[       rowId_child  x_child  y_child  z_child  radius_child  link_child  \
 0                2  19752.0  16724.0  18778.0     10.970600           1   
 1                3  19752.0  16700.0  18754.0     18.000000           2   
 2                4  19740.0  16688.0  18742.0     10.970600           3   
 3                5  19716.0  16676.0  18742.0     18.000000           4   
 4                6  19692.0  16676.0  18730.0     18.000000           5   
 ...            ...      ...      ...      ...           ...         ...   
 15392        15397  15396.0  12884.0  25774.0     10.970600       15396   
 15393        15398  15384.0  12884.0  25774.0      6.000000       15397   
 15394        15399  15864.0  12788.0  25546.0     27.941099       15376   
 15395        15400  15864.0  12896.0  25510.0     10.970600       15375   
 15396        15401  15876.0  12908.0  25522.0      6.000000       15400   
 
            bodyId color_child  rowId_parent      x_parent      y_parent  \
 0      10

In [ ]:
# plotting using bokeh (usually takes around 15 secs to render)
pmpre = figure(width=500, height=450, output_backend='webgl') 
pmpre.y_range.flipped = True

for segments in skel[:]:
        # Plot skeleton segments (in 2D)
    pmpre.segment(x0='x_child', x1='x_parent',
                y0='z_child', y1='z_parent',
                color='color_child',
                source=segments)
        
pmpre.xaxis.visible = False
pmpre.xgrid.visible = False

pmpre.yaxis.visible = False
pmpre.ygrid.visible = False
    # no gray outline
pmpre.outline_line_color = None
output_file("large_plot.html", mode='inline')
#show(pmpre)